# Anveshak: Open-World Remote Sensing Object Detector
### Techfest 2026-27 | IITB-HRTI Centre of Excellence
---
This turnkey notebook runs GPU training, continual few-shot learning, and evaluation directly inside Google Colab.
- **Dataset (20 GB):** Extracted from your Google Drive shortcut directly into local fast SSD.
- **Outputs:** Model checkpoints and `predictions.csv` are automatically saved to your Google Drive.

## 1. Verify GPU & Environment Setup

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_ANVESHAK_DIR = '/content/drive/MyDrive/Anveshak'
os.makedirs(f"{DRIVE_ANVESHAK_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_ANVESHAK_DIR}/submissions", exist_ok=True)
print("[✓] Google Drive connected successfully. Checkpoints will be saved here.")

## 3. Setup Project Codebase from GitHub

In [ ]:
import os
import shutil

GITHUB_REPO_URL = "https://github.com/Likhitha-talent/IITB-Anveshak.git"
REPO_NAME = "IITBBB"

%cd /content
if not os.path.exists(REPO_NAME):
    !git clone {GITHUB_REPO_URL} {REPO_NAME}
else:
    %cd {REPO_NAME}
    !git pull

%cd /content/{REPO_NAME}
!pip install -r requirements.txt -q
!pip install gdown -q

# Set PYTHONPATH
os.environ['PYTHONPATH'] = f"/content/{REPO_NAME}"
print("[✓] Codebase updated and ready in:", os.getcwd())

## 4. Extract Round 1 Dataset (20 GB) Directly to Colab SSD

In [ ]:
import os
import zipfile

!mkdir -p /content/data

possible_paths = [
    "/content/drive/MyDrive/Anveshak/round1.zip",
    "/content/drive/MyDrive/round1.zip",
    "/content/drive/MyDrive/Anveshak/datasets/round1.zip"
]

zip_path = None
for p in possible_paths:
    if os.path.exists(p):
        zip_path = p
        break

if not zip_path:
    matches = !find /content/drive/MyDrive -name "*round1*" 2>/dev/null
    if matches:
        zip_path = matches[0]

if os.path.exists("/content/data/round1"):
    print("[✓] Dataset already extracted in /content/data/round1:")
    !ls -lh /content/data/round1
else:
    print(f"[*] Found dataset at: {zip_path}")
    print("[*] Extracting directly into fast local SSD (/content/data/)... Please wait ~2-3 mins.")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("/content/data/")
    print("[✓] Round 1 Dataset successfully extracted! Folders ready:")
    !ls -lh /content/data

## 5. Train Detector on Set A (From Scratch - Random Initialization)

In [ ]:
%cd /content/IITBBB
# Execute Set A training script
!PYTHONPATH=/content/IITBBB python -m src.training.train_set_a \
    --config configs/base_config.yaml \
    --data_dir /content/data \
    --output_dir checkpoints/

# Back up Set A weights to Google Drive immediately
!cp checkpoints/set_a_best.pth {DRIVE_ANVESHAK_DIR}/checkpoints/
print("[✓] Set A checkpoint saved and synced to Drive.")

## 6. Continual Few-Shot Learning on Set B
Continues the *exact same model lineage* on Set B (10 new classes) with distillation loss and cosine classifier margin to retain Set A classes.

In [ ]:
%cd /content/IITBBB
!PYTHONPATH=/content/IITBBB python -m src.training.train_set_b \
    --config configs/base_config.yaml \
    --checkpoint_in checkpoints/set_a_best.pth \
    --data_dir /content/data \
    --output_checkpoint checkpoints/round1_final.pth

# Sync final checkpoint to Google Drive
!cp checkpoints/round1_final.pth {DRIVE_ANVESHAK_DIR}/checkpoints/
print("[✓] Round 1 Checkpoint saved and backed up to Google Drive.")

## 7. Inference & Official Conflict Rule Resolution
Generates raw predictions on unlabelled Round 1 test images, applies conflict filtering (IoU >= 0.5 highest score rule), and exports `predictions.csv`.

In [ ]:
%cd /content/IITBBB
!PYTHONPATH=/content/IITBBB python -m src.inference.predict \
    --checkpoint checkpoints/round1_final.pth \
    --test_images_dir /content/data/test_images \
    --output_csv submission/predictions.csv

# Copy predictions to Google Drive
!cp submission/predictions.csv {DRIVE_ANVESHAK_DIR}/submissions/
print("[✓] Submission CSV saved and synced to Drive.")

## 8. Validate Predictions with Organizer's Script

In [ ]:
import os

# Locate the organizer's validate_predictions.py inside /content/data
validator_path = None
for root, dirs, files in os.walk("/content/data"):
    if "validate_predictions.py" in files:
        validator_path = os.path.join(root, "validate_predictions.py")
        break

if validator_path:
    print(f"[*] Found validator script at: {validator_path}")
    !python "{validator_path}" /content/IITBBB/submission/predictions.csv
else:
    print("[!] validate_predictions.py not found in /content/data/")